##**Fake News Classifier Using Bidirectional LSTM**
Dataset: https://www.kaggle.com/c/fake-news/data#

In [149]:
import pandas as pd

In [150]:
news=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train.txt', on_bad_lines='skip', engine='python')
news.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [151]:
###Drop Nan Values
news=news.dropna()

In [152]:
## Get the Independent Features

X=news.drop('label',axis=1)

In [153]:
## Get the Dependent features
y=news['label']

In [154]:
y=news['label'].unique()
print(y)

['1' '0' ' как люди воспринимают своё положение.'
 ' чтобы это была дорога с двусторонним движением.']


**Corrupted Dataset**

In [155]:
# Keep only rows where label is '0' or '1'
news = news[news['label'].isin(['0', '1'])]

# Now convert label to integer
news['label'] = news['label'].astype(int)

In [156]:
X=news.drop('label',axis=1)

In [157]:
y = news['label']

In [158]:
print(news['label'].unique())

[1 0]


In [159]:
y.value_counts()

,count
label,
0,10361
1,7922


In [160]:
X.shape

(18283, 4)

In [161]:
y.shape

(18283,)

In [162]:
import tensorflow as tf

In [163]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.layers import Dense

In [164]:
### Vocabulary size
voc_size=5000

###**Onehot Representation**

In [165]:
messages=X.copy()

In [166]:
messages['title'][25]

'Nukes and the UN: a Historic Treaty to Ban Nuclear Weapons'

In [167]:
messages.reset_index(inplace=True)

In [168]:
import nltk
import re
from nltk.corpus import stopwords

In [169]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [170]:
### Dataset Preprocessing
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
    print(i)
    review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
    review = review.lower()
    review = review.split()

    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

Streaming output truncated to the last 5000 lines.
13283
13284
13285
13286
13287
13288
13289
13290
13291
13292
13293
13294
13295
13296
13297
13298
13299
13300
13301
13302
13303
13304
13305
13306
13307
13308
13309
13310
13311
13312
13313
13314
13315
13316
13317
13318
13319
13320
13321
13322
13323
13324
13325
13326
13327
13328
13329
13330
13331
13332
13333
13334
13335
13336
13337
13338
13339
13340
13341
13342
13343
13344
13345
13346
13347
13348
13349
13350
13351
13352
13353
13354
13355
13356
13357
13358
13359
13360
13361
13362
13363
13364
13365
13366
13367
13368
13369
13370
13371
13372
13373
13374
13375
13376
13377
13378
13379
13380
13381
13382
13383
13384
13385
13386
13387
13388
13389
13390
13391
13392
13393
13394
13395
13396
13397
13398
13399
13400
13401
13402
13403
13404
13405
13406
13407
13408
13409
13410
13411
13412
13413
13414
13415
13416
13417
13418
13419
13420
13421
13422
13423
13424
13425
13426
13427
13428
13429
13430
13431
13432
13433
13434
13435
13436
13437
13438
13439
13440
1

In [171]:
corpus

['hous dem aid even see comey letter jason chaffetz tweet',
 'flynn hillari clinton big woman campu breitbart',
 'truth might get fire',
 'civilian kill singl us airstrik identifi',
 'iranian woman jail fiction unpublish stori woman stone death adulteri',
 'jacki mason hollywood would love trump bomb north korea lack tran bathroom exclus video breitbart',
 'beno hamon win french socialist parti presidenti nomin new york time',
 'back channel plan ukrain russia courtesi trump associ new york time',
 'obama organ action partner soro link indivis disrupt trump agenda',
 'bbc comedi sketch real housew isi caus outrag',
 'russian research discov secret nazi militari base treasur hunter arctic photo',
 'us offici see link trump russia',
 'ye paid govern troll social media blog forum websit',
 'major leagu soccer argentin find home success new york time',
 'well fargo chief abruptli step new york time',
 'anonym donor pay million releas everyon arrest dakota access pipelin',
 'fbi close hilla

In [172]:
onehot_repr=[one_hot(words,voc_size)for words in corpus]
onehot_repr

[[4460, 1897, 3919, 3696, 691, 1422, 3533, 1446, 2036, 2817],
 [2148, 3075, 4358, 139, 3631, 4786, 133],
 [3337, 2179, 191, 1640],
 [1188, 969, 3346, 3648, 357, 3732],
 [3396, 3631, 3224, 2596, 1114, 1871, 3631, 4213, 3402, 1952],
 [153,
  2791,
  4554,
  1050,
  2926,
  1155,
  3539,
  1880,
  2751,
  2236,
  4532,
  4880,
  4348,
  3383,
  133],
 [3002, 4244, 3362, 4810, 3963, 4067, 4206, 4016, 4301, 2663, 818],
 [3850, 1533, 1531, 1163, 61, 946, 1155, 2999, 4301, 2663, 818],
 [779, 148, 3071, 2657, 1746, 733, 3039, 4591, 1155, 325],
 [1983, 1558, 2516, 3868, 1223, 2108, 3720, 389],
 [1993, 2257, 1870, 126, 182, 877, 4536, 2302, 1942, 2401, 1866],
 [3648, 4814, 691, 733, 1155, 61],
 [65, 3339, 2928, 2703, 2765, 3028, 2585, 3759, 3360],
 [670, 2212, 4713, 3529, 151, 3764, 753, 4301, 2663, 818],
 [2582, 1359, 920, 4540, 1858, 4301, 2663, 818],
 [4191, 722, 3809, 3236, 624, 4429, 2787, 4103, 3011, 2527],
 [924, 1349, 3075],
 [1755, 4771, 2927, 1586, 1155, 2227, 2090, 133],
 [1947, 2947,

###**Embedding Representation**

In [173]:
sent_length=20
embedded_docs=pad_sequences(onehot_repr,padding='pre',maxlen=sent_length)
print(embedded_docs)

[[   0    0    0 ... 1446 2036 2817]
 [   0    0    0 ... 3631 4786  133]
 [   0    0    0 ... 2179  191 1640]
 ...
 [   0    0    0 ... 4301 2663  818]
 [   0    0    0 ... 3358 3938 2769]
 [   0    0    0 ... 3217 2708 2737]]


In [174]:
embedded_docs[0]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0, 4460,
       1897, 3919, 3696,  691, 1422, 3533, 1446, 2036, 2817], dtype=int32)

In [175]:
## Creating model
embedding_vector_features=40
model1=Sequential()
model1.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model1.add(Bidirectional(LSTM(100)))
model1.add(Dropout(0.3))
model1.add(Dense(1,activation='sigmoid'))
model1.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [176]:
len(embedded_docs),y.shape

(18283, (18283,))

In [177]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [178]:
X_final.shape,y_final.shape

((18283, 20), (18283,))

In [179]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

###**Model Training**

In [180]:
### Finally Training
model1.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 20s 75ms/step - accuracy: 0.7805 - loss: 0.4200 - val_accuracy: 0.9118 - val_loss: 0.2024
Epoch 2/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 16s 85ms/step - accuracy: 0.9484 - loss: 0.1332 - val_accuracy: 0.9213 - val_loss: 0.1932
Epoch 3/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 16s 81ms/step - accuracy: 0.9691 - loss: 0.0907 - val_accuracy: 0.9181 - val_loss: 0.2399
Epoch 4/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 16s 83ms/step - accuracy: 0.9806 - loss: 0.0594 - val_accuracy: 0.9156 - val_loss: 0.2357
Epoch 5/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 20s 82ms/step - accuracy: 0.9865 - loss: 0.0399 - val_accuracy: 0.9087 - val_loss: 0.3425
Epoch 6/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 19s 75ms/step - accuracy: 0.9925 - loss: 0.0275 - val_accuracy: 0.9128 - val_loss: 0.3992
Epoch 7/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 21s 77ms/step - accuracy: 0.9949 - loss: 0.0167 - val_accuracy: 0.9115 - val_loss: 0.4073
Epoch 8/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 15s 76ms/step - accuracy: 0.9951 - loss: 0.0138 - 

###**Performance Metrics And Accuracy**

In [181]:
y_pred1=model1.predict(X_test)

189/189 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step


In [182]:
# Use the predict method to get the probability outputs
y_pred_prob1 = model1.predict(X_test)

# Convert the probabilities to class labels using a threshold (0.5 for sigmoid)
y_pred1 = (y_pred_prob1 > 0.5).astype(int)

189/189 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step


In [183]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test,y_pred1)

array([[3101,  314],
       [ 239, 2380]])

In [184]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred1)

0.9083526682134571

In [185]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred1))

              precision    recall  f1-score   support

           0       0.93      0.91      0.92      3415
           1       0.88      0.91      0.90      2619

    accuracy                           0.91      6034
   macro avg       0.91      0.91      0.91      6034
weighted avg       0.91      0.91      0.91      6034



**Consider LSTM**

In [186]:
## Creating model
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [187]:
### Finally Training
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

y_pred=model.predict(X_test)

# Use the predict method to get the probability outputs
y_pred_prob = model.predict(X_test)

# Convert the probabilities to class labels using a threshold (0.5 for sigmoid)
y_pred = (y_pred_prob > 0.5).astype(int)

Epoch 1/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 13s 55ms/step - accuracy: 0.7855 - loss: 0.4226 - val_accuracy: 0.9090 - val_loss: 0.2104
Epoch 2/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 8s 40ms/step - accuracy: 0.9431 - loss: 0.1382 - val_accuracy: 0.9165 - val_loss: 0.1925
Epoch 3/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 10s 51ms/step - accuracy: 0.9668 - loss: 0.0905 - val_accuracy: 0.9151 - val_loss: 0.2029
Epoch 4/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.9799 - loss: 0.0578 - val_accuracy: 0.9203 - val_loss: 0.2405
Epoch 5/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - accuracy: 0.9873 - loss: 0.0414 - val_accuracy: 0.9185 - val_loss: 0.2822
Epoch 6/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 11s 51ms/step - accuracy: 0.9893 - loss: 0.0332 - val_accuracy: 0.9065 - val_loss: 0.3218
Epoch 7/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - accuracy: 0.9943 - loss: 0.0183 - val_accuracy: 0.9115 - val_loss: 0.3559
Epoch 8/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 8s 43ms/step - accuracy: 0.9973 - loss: 0.0106 - val_

In [188]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test,y_pred)

array([[3098,  317],
       [ 227, 2392]])

In [189]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.9098442161087172

In [190]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.93      0.91      0.92      3415
           1       0.88      0.91      0.90      2619

    accuracy                           0.91      6034
   macro avg       0.91      0.91      0.91      6034
weighted avg       0.91      0.91      0.91      6034

